# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

EXTRA notebook, not presented in class: **the algebra behind Day 10** - generalized least squares, attenuation and the generative model's likelihood. Read alongside Day 10 and Lab 04.

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Generalized least squares

Tuesday's design matrix $A$ has one row per data point and one column per parameter, so the model is $\mathbf{y} = A\boldsymbol{\theta} + \boldsymbol{\epsilon}$. For a straight line, row $i$ is $(1, x_i)$.

Assumption 3, independent points, is a diagonal $\Sigma$. The algebra never needed it:

$$\hat{\boldsymbol{\theta}} = (A^{\mathsf T}\Sigma^{-1}A)^{-1}A^{\mathsf T}\Sigma^{-1}\mathbf{y}, \qquad \mathrm{Cov}(\hat{\boldsymbol{\theta}}) = (A^{\mathsf T}\Sigma^{-1}A)^{-1}, \qquad \Sigma_{ij} = \mathrm{Cov}(\epsilon_i, \epsilon_j)$$

* $\mathrm{Cov}(a, b) = E[(a - E[a])(b - E[b])]$ says how much two noisy numbers move together: $\mathrm{Cov}(a, a) = \mathrm{Var}(a)$, $\mathrm{Cov}(a, ka) = k\,\mathrm{Var}(a)$, and independent pieces add nothing
* with a diagonal $\Sigma$ this is WLS; with $\Sigma = \sigma^2 I$ it is OLS
* $\sigma \propto 1/\sqrt{N}$, so an error bar 4$\times$ too small means the fit counted $4^2 = 16\times$ too many independent points

## Attenuation, derived

* true $x \sim \mathcal{N}(\mu, \sigma_{\rm pop}^2)$, with $\mathcal{N}$ written as (mean, variance)
* we observe $x_{\textrm{obs}} = x_{\textrm{true}} + \epsilon_x$ with $\epsilon_x \sim \mathcal{N}(0, \sigma_x^2)$, and $y = \alpha\,x_{\textrm{true}} + \beta + \epsilon_y$
* centre $x$ and $y$, and the 2$\times$2 normal equations give slope $= \sum x y / \sum x^2 = \mathrm{Cov}(x_{\textrm{obs}}, y)/\mathrm{Var}(x_{\textrm{obs}})$
* $\epsilon_x$ has nothing to do with $y$, so the top stays $\alpha\sigma_{\rm pop}^2$ and only the bottom grows, to $\sigma_{\rm pop}^2 + \sigma_x^2$:

$$E[\hat\alpha_{\textrm{OLS}}] = \alpha\,\frac{\sigma_{\rm pop}^2}{\sigma_{\rm pop}^2 + \sigma_x^2}$$

* set by $\sigma_x/\sigma_{\rm pop}$ alone, so more data does not help

## The generative model, as equations

* $x_{\textrm{true}} \sim \mathcal{N}(\mu, \sigma_{\rm pop}^2)$
* $y_{\textrm{true}} = \alpha(x_{\textrm{true}} - 9.5) + \beta + \mathcal{N}(0, \sigma_{\textrm{int}}^2)$, pivoted at $10^{9.5}\,M_\odot$, like the Cepheids at 10 days, so $\alpha$ and $\beta$ are less correlated
* $x_i = x_{\textrm{true},i} + \mathcal{N}(0, \sigma_{x,i}^2)$ and $y_i = y_{\textrm{true},i} + \mathcal{N}(0, \sigma_{y,i}^2)$
* integrate out (marginalize) the latent $x_{\textrm{true}}$:

$$p(x_i, y_i) = \int p(x_i, y_i \mid x_{\textrm{true},i})\; p(x_{\textrm{true},i} \mid \mu, \sigma_{\rm pop})\; dx_{\textrm{true},i}$$

* the Cepheid fit was the case with no scatter and exact periods - each true magnitude was on the line, a $\delta$-function
* the mixture's nuisance parameters were taken at their best fit, and here we integrate - Sep 29 comes back to that difference

## The marginal likelihood

* do the integral, and each observed pair is one draw from a bivariate Gaussian (a 2-D bell curve with a 2$\times$2 covariance)
* the variances add because the pieces are independent, and $\mathrm{Cov}(x_i, y_i) = \mathrm{Cov}(x_{\textrm{true}}, \alpha x_{\textrm{true}}) = \alpha\sigma_{\rm pop}^2$, the attenuation numerator again

$$\begin{pmatrix} x_i \\ y_i \end{pmatrix} \sim \mathcal{N}\!\left[\begin{pmatrix} \mu \\ \alpha(\mu - 9.5) + \beta \end{pmatrix},\;\begin{pmatrix} \sigma_{\rm pop}^2 + \sigma_{x,i}^2 & \alpha\sigma_{\rm pop}^2 \\ \alpha\sigma_{\rm pop}^2 & \alpha^2\sigma_{\rm pop}^2 + \sigma_{\textrm{int}}^2 + \sigma_{y,i}^2 \end{pmatrix}\right]$$

* 5 parameters ($\alpha$, $\beta$, $\sigma_{\textrm{int}}$, plus $\mu$, $\sigma_{\rm pop}$ for the population)
* an off-diagonal term again, as in GLS, but now between the $x$ and $y$ of one point: it says how much of the spread in $x$ is real
* give every $x_{\textrm{true}}$ a flat prior instead (no population, $\sigma_{\rm pop} \to \infty$) and you get (nearly) OLS back, attenuation and all (Kelly 2007, Sec. 4.2)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The off-diagonal $\alpha\sigma_{\rm pop}^2$ is the correction!**

</div>

## Writing the likelihood

For each point, $\mathbf{d}_i$ is its offset from the mean and $C_i$ the 2$\times$2 matrix above:

$$\mathbf{d}_i = \begin{pmatrix} x_i - \mu \\ y_i - \alpha(\mu - 9.5) - \beta \end{pmatrix}, \qquad -2\ln L = \sum_i \left[\ln\det C_i + \mathbf{d}_i^{\mathsf T} C_i^{-1}\mathbf{d}_i\right] + \textrm{const}$$

* $\mathbf{d}^{\mathsf T}C^{-1}\mathbf{d}$ is the same $\chi^2$ as always
* $\ln\det C_i$ is there because $C_i$ now depends on the parameters
* fit $\ln\sigma_{\textrm{int}}$ and $\ln\sigma_{\rm pop}$, so the widths stay positive
* maximize numerically, error bars from the curvature - Lab 04 has you do it